# Synthesize Dubins-car control laws (unconstrained + constrained)

This notebook drives the MATLAB reach-avoid backstepping synthesis to generate
**both** Dubins-car control laws, then loads and displays them:

- **unconstrained** (vanilla baseline) -> `controllers/sop_bounded_control_dubins_car_unconstrained.py`
- **constrained** (bounded-control SOP result) -> `controllers/sop_bounded_control_dubins_car_result.py`

It runs the canonical, unmodified [`matlab/example_dubins_car.m`](../matlab/example_dubins_car.m)
via `matlab -batch` (reusing `python/matlab_runner.py`). A single MATLAB run produces
both control laws.

## Prerequisites

- MATLAB on `PATH` with **SOSTOOLS + Mosek** installed.
- Run with the **`rab_mpc`** kernel (provides `sympy`).
- The SOP solve takes a few minutes.


In [ ]:
# -- Repo-root bootstrap -------------------------------------------------------
# Make ../controllers and ../python importable and locate ../matlab, regardless
# of whether this notebook is launched from notebooks/ or the repository root.
import os
import sys

ROOT = os.getcwd()
for _ in range(4):
    if os.path.isdir(os.path.join(ROOT, "controllers")) and os.path.isdir(
        os.path.join(ROOT, "matlab")
    ):
        break
    ROOT = os.path.dirname(ROOT)

for _sub in ("controllers", "python"):
    _p = os.path.join(ROOT, _sub)
    if _p not in sys.path:
        sys.path.insert(0, _p)

MATLAB_DIR = os.path.join(ROOT, "matlab")
CONTROLLERS_DIR = os.path.join(ROOT, "controllers")
print("ROOT            =", ROOT)
print("MATLAB_DIR      =", MATLAB_DIR)
print("CONTROLLERS_DIR =", CONTROLLERS_DIR)
assert os.path.isdir(MATLAB_DIR), f"matlab/ not found under {ROOT}"

In [ ]:
# -- Locate the MATLAB executable ---------------------------------------------
from matlab_runner import find_matlab, run_one

# Honour PATH first; fall back to the standard install location on this machine.
MATLAB = find_matlab() or find_matlab("/usr/local/bin/matlab")
if MATLAB is None:
    raise RuntimeError(
        "MATLAB executable not found. Install MATLAB (with SOSTOOLS + Mosek) or add "
        "it to PATH, e.g. `export PATH=$PATH:/usr/local/bin`."
    )
print("MATLAB =", MATLAB)

In [ ]:
# -- Run the MATLAB synthesis (this is the multi-minute step) -----------------
# example_dubins_car.m writes BOTH controllers into controllers/ in one run.
EXPORTS = {
    "unconstrained": os.path.join(
        CONTROLLERS_DIR, "sop_bounded_control_dubins_car_unconstrained.py"
    ),
    "constrained": os.path.join(
        CONTROLLERS_DIR, "sop_bounded_control_dubins_car_result.py"
    ),
}

# Snapshot pre-run mtimes so we can confirm the files were (re)written by this run.
_before = {
    k: (os.path.getmtime(p) if os.path.exists(p) else None) for k, p in EXPORTS.items()
}

result = run_one(MATLAB, MATLAB_DIR, "example_dubins_car")

print(f"status      = {result['status']}")
print(f"design_s    = {result['design_s']}")
print(f"sop_solve_s = {result['sop_solve_s']}")
print(f"wall_s      = {result['wall_s']:.1f}")
print("-" * 70)
print("MATLAB stdout (tail):")
print("\n".join(result["stdout"].splitlines()[-25:]))

if result["status"] != "ok":
    raise RuntimeError(
        f"MATLAB synthesis failed ({result['status']}): {result['error']}\n"
        f"stderr:\n{result['stderr']}"
    )

In [ ]:
# -- Confirm both control-law files were freshly written ----------------------
import datetime as _dt

for _kind, _path in EXPORTS.items():
    assert os.path.exists(_path), f"Expected export missing: {_path}"
    _mtime = os.path.getmtime(_path)
    _fresh = (_before[_kind] is None) or (_mtime > _before[_kind])
    _stamp = _dt.datetime.fromtimestamp(_mtime).strftime("%Y-%m-%d %H:%M:%S")
    print(
        f"{_kind:13s} {'(new) ' if _fresh else '(STALE)'}  "
        f"{os.path.getsize(_path):6d} B  {_stamp}  {_path}"
    )
    assert _fresh, f"{_path} was not updated by this run (stale)."

In [ ]:
# -- Record per-stage + grouped (unconstrained/constrained) timing ------------
# example_dubins_car.m / solvesop_bounded_control.m emit one
# `__TIMING__,<script>,<phase>,<seconds>` line per stage; parse_timings collects them.
from matlab_runner import parse_timings

# Which controller each stage contributes to. `design` is shared symbolic groundwork
# for both; the unconstrained (vanilla) controller is produced first, the constrained
# bounded-control controller after it.
_GROUPS = {
    "shared": ["design"],
    "unconstrained": ["vanilla_solve", "uncon_export"],
    "constrained": [
        "sampling",
        "sop_k1_solve",
        "final_subs",
        "bounds_estimation",
        "result_export",
    ],
}
_STAGE_ORDER = [s for stages in _GROUPS.values() for s in stages]

_phases = parse_timings(result["stdout"])  # every __TIMING__ marker
_total = float(
    _phases.get("total", result.get("wall_s"))
)  # MATLAB total, else wall clock

# Flat per-stage dict (umbrella 'sop_solve' and 'total' excluded); plus group subtotals.
_stage_timing = {n: _phases[n] for n in _STAGE_ORDER if n in _phases}
for _n, _v in _phases.items():  # surface any unexpected extra phase
    if _n not in _stage_timing and _n not in ("sop_solve", "total"):
        _stage_timing[_n] = _v
_subtotal = {
    g: sum(_phases.get(s, 0.0) for s in stages) for g, stages in _GROUPS.items()
}

# Grouped timing table.
print(f"{'group':<14} {'stage':<20} {'seconds':>10}")
print("-" * 46)
for _g, _stages in _GROUPS.items():
    for _s in _stages:
        if _s in _phases:
            print(f"{_g:<14} {_s:<20} {_phases[_s]:>10.3f}")
    print(f"{'':<14} {_g + ' subtotal':<20} {_subtotal[_g]:>10.3f}")
    print("-" * 46)
print(f"{'':<14} {'TOTAL':<20} {_total:>10.3f}")


def embed_timings(py_path, stage_timing, groups, subtotal, total_s):
    """Append (idempotently) the timing + grouped subtotals to an exported .py."""
    marker = "# === stage timings (seconds, auto-generated) ==="
    text = open(py_path).read()
    cut = text.find(marker)
    if cut != -1:  # drop any previous block first
        text = text[:cut].rstrip() + "\n"
    L = ["", "", marker, "timing = {"]
    L += [f"    {k!r}: {v}," for k, v in stage_timing.items()]
    L += ["}", "timing_groups = {"]
    L += [f"    {g!r}: {stages!r}," for g, stages in groups.items()]
    L += [
        "}",
        f"timing_shared_s = {subtotal['shared']}",
        f"timing_unconstrained_s = {subtotal['unconstrained']}",
        f"timing_constrained_s = {subtotal['constrained']}",
        f"timing_total_s = {total_s}",
        "",
    ]
    with open(py_path, "w") as f:
        f.write(text.rstrip() + "\n" + "\n".join(L))


for _kind, _path in EXPORTS.items():
    embed_timings(_path, _stage_timing, _GROUPS, _subtotal, _total)
    print("embedded timing ->", _path)

In [ ]:
# -- Load + display the synthesized control laws ------------------------------
import importlib

import sympy as sp
from IPython.display import display


def load_controller(module_name):
    """Import (or reload) a freshly written controller module from controllers/."""
    importlib.invalidate_caches()
    if module_name in sys.modules:
        return importlib.reload(sys.modules[module_name])
    return importlib.import_module(module_name)


def _param_block(path):
    """Extract the `# name = value` parameter comments from an export.

    Only lines whose comment body starts with an identifier are kept, so the
    auto-generated `# === stage timings ... ===` marker is not picked up.
    """
    out = []
    with open(path) as f:
        for line in f:
            s = line.strip()
            if s.startswith("# ") and "=" in s:
                body = s[2:].strip()
                name = body.split("=", 1)[0].strip()
                if name[:1].isalpha() or name[:1] == "_":
                    out.append(body)
    return out


def show_controller(title, module_name, path):
    """Print a summary and render u_opt / k1_opt / certificate_opt for one export."""
    mod = load_controller(module_name)
    u = sp.Matrix(mod.u_opt)  # control law in state space (omega, a)
    k1 = sp.Matrix(mod.k1_opt)  # virtual control in output (y) space
    cert = sp.sympify(mod.certificate_opt)
    print("=" * 72)
    print(title)
    print("module :", module_name)
    print("params :", ", ".join(_param_block(path)) or "(none)")
    print("u_opt   free symbols:", u.free_symbols)
    print("k1_opt  free symbols:", k1.free_symbols)
    print("cert    free symbols:", cert.free_symbols)
    print(
        "sizes  : u_opt ~%d chars, k1_opt ~%d chars, certificate ~%d chars"
        % (len(str(u)), len(str(k1)), len(str(cert)))
    )
    if hasattr(mod, "timing_total_s"):
        print(
            "timing : unconstrained %.3fs + constrained %.3fs + shared %.3fs = total %.3fs"
            % (
                getattr(mod, "timing_unconstrained_s", float("nan")),
                getattr(mod, "timing_constrained_s", float("nan")),
                getattr(mod, "timing_shared_s", float("nan")),
                mod.timing_total_s,
            )
        )
    print("\nu_opt = [omega; a] :")
    display(u)
    print("k1_opt (output-space virtual control) :")
    display(k1)
    print("certificate_opt (certified set is {certificate_opt >= 0}) :")
    display(cert)
    return mod, u, k1, cert


# --- Unconstrained (vanilla baseline) ---
uncon = show_controller(
    "UNCONSTRAINED (vanilla baseline)",
    "sop_bounded_control_dubins_car_unconstrained",
    EXPORTS["unconstrained"],
)

In [ ]:
# --- Constrained (bounded-control SOP result) ---
con = show_controller(
    "CONSTRAINED (bounded-control SOP result)",
    "sop_bounded_control_dubins_car_result",
    EXPORTS["constrained"],
)

## Summary

Both control laws come from the **same** MATLAB run of `example_dubins_car.m`:

- **Unconstrained** -- the vanilla backstepping `k1` controller solved **without** input-bound
  enforcement (`solve_vanilla_k1_controller`); the baseline closed-form law.
- **Constrained** -- the bounded-control **SOP** result (`solve_k1_controller_sop`), which enforces
  the input bounds `|omega| <= 5`, `|a| <= 5` over the sampled state set.

Each exported certificate is shifted by `-delta/lambda` (the certified reach-avoid set is
`{certificate_opt >= 0}`), with the solved `delta` substituted numerically, so no free `delta`
symbol remains in the exports.

### Stage timings (split by controller)

Each exported `.py` ends with an auto-generated `timing = {...}` dict (seconds per stage),
a `timing_groups` map, and grouped subtotals `timing_shared_s` / `timing_unconstrained_s` /
`timing_constrained_s` / `timing_total_s`. Stages are grouped as:

- **shared**: `design` (symbolic backstepping, needed by both controllers)
- **unconstrained**: `vanilla_solve`, `uncon_export`
- **constrained**: `sampling`, `sop_k1_solve`, `final_subs`, `bounds_estimation`, `result_export`

They come from `__TIMING__` markers emitted by the MATLAB code and are parsed/embedded by the
timing cell above. (`design` is shared groundwork, so it is reported separately rather than
charged to either controller.)
